In [13]:
#step 1: import library yang diperlukan
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

housing = pd.read_csv("housing.csv")

# buat kategori berdasarkan median_income agar tes set sesuai dengan proporsi ditribusi median_income
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5],
)

# buat data train dan test dengan stratified sampling berdasarkan income_cat
train_set, test_set = train_test_split(
    housing, test_size=0.2, stratify=housing["income_cat"], random_state=42
)

# cek hasil proporsi income_cat di test set dan train set
test_set["income_cat"].value_counts() / len(test_set)
train_set["income_cat"].value_counts() / len(train_set)


# hapus kolom income_cat karena sudah tidak diperlukan lagi
for set_ in (train_set, test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing=train_set.copy()
housing_train_x= train_set.drop("median_house_value", axis=1)
housing_labels_train_y = train_set["median_house_value"].copy()

In [14]:
#buat class untuk data preparation untuk model clustering
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted, check_array
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel


class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(n_clusters=self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self #always return self in fit method
    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
    def get_feature_names_out(self, names=None):
        return [f"cluster_similarity_{i}" for i in range(self.n_clusters)]

In [15]:
#buat pipeline untuk data preparation
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def column_ratio(X):
    return X[:, [0]] / X[:, [1]]
def ratio_names(function_transformer, feature_names_in):
    return ["ratio"]
def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_names),
        StandardScaler(),
    )

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler())

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)
    
cluster_simil= ClusterSimilarity(n_clusters=10, gamma=1.0, random_state=42)

default_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)   

preprocessing = ColumnTransformer([
    ("bedrooms_per_room", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_household", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_household", ratio_pipeline(), ["population", "households"]),
    ("log", log_pipeline, ["total_bedrooms", "total_rooms", "households", 
                           "population", "median_income"]),
    ("geo", cluster_simil, ["latitude", "longitude"]),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
],
    remainder=default_pipeline)





In [16]:
housing_prepared = preprocessing.fit_transform(housing_train_x)
housing_prepared.shape

(16512, 24)

In [17]:
preprocessing.get_feature_names_out()

array(['bedrooms_per_room__ratio', 'rooms_per_household__ratio',
       'people_per_household__ratio', 'log__total_bedrooms',
       'log__total_rooms', 'log__households', 'log__population',
       'log__median_income', 'geo__cluster_similarity_0',
       'geo__cluster_similarity_1', 'geo__cluster_similarity_2',
       'geo__cluster_similarity_3', 'geo__cluster_similarity_4',
       'geo__cluster_similarity_5', 'geo__cluster_similarity_6',
       'geo__cluster_similarity_7', 'geo__cluster_similarity_8',
       'geo__cluster_similarity_9', 'cat__ocean_proximity_<1H OCEAN',
       'cat__ocean_proximity_INLAND', 'cat__ocean_proximity_ISLAND',
       'cat__ocean_proximity_NEAR BAY', 'cat__ocean_proximity_NEAR OCEAN',
       'remainder__housing_median_age'], dtype=object)

In [18]:
#select and train a model pertama kita coba linear regression dulu
from sklearn.linear_model import LinearRegression
lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing_train_x, housing_labels_train_y)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                              SimpleImputer(strategy='median')),
                                                             ('standardscaler',
                                                              StandardScaler())]),
                                   transformers=[('bedrooms_per_room',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('functiontransformer',
                                                                   FunctionTransformer(feature_names_out=<function ratio_na...
                                                   'median_income']),
                                                 ('geo',
                                                  ClusterSimilarity(random_state=42),
                                                  ['latitude', 'longitude']),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x798f4b48ae10>)])),
                ('linearregression', LinearRegression())])

In [19]:
#hasil prediksi dengan linear regression
housing_predictions_lin = lin_reg.predict(housing_train_x)
print(housing_predictions_lin[:5].round(-2))
print(housing_labels_train_y[:5].values)
#rmse
from sklearn.metrics import root_mean_squared_error
lin_rmse = root_mean_squared_error(housing_labels_train_y, housing_predictions_lin)
print(f"Linear Regression RMSE sebesar: {lin_rmse}")

[246000. 372700. 135700.  91400. 330900.]
[458300. 483800. 101700.  96100. 361800.]
Linear Regression RMSE sebesar: 68972.88910758484


In [20]:
#model kedua kita coba decision tree regressor
from sklearn.tree import DecisionTreeRegressor
tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing_train_x, housing_labels_train_y)



Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                              SimpleImputer(strategy='median')),
                                                             ('standardscaler',
                                                              StandardScaler())]),
                                   transformers=[('bedrooms_per_room',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('functiontransformer',
                                                                   FunctionTransformer(feature_names_out=<function ratio_na...
                                                  ClusterSimilarity(random_state=42),
                                                  ['latitude', 'longitude']),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x798f4b48ae10>)])),
                ('decisiontreeregressor',
                 DecisionTreeRegressor(random_state=42))])

In [21]:
#buat prediksi dengan decision tree regressor
housing_predictions_tree = tree_reg.predict(housing_train_x)
#hitung rmse untuk decision tree regressor
from sklearn.metrics import root_mean_squared_error
tree_rmse = root_mean_squared_error(housing_labels_train_y, housing_predictions_tree)
print(housing_predictions_tree[:5].round(-2))
print(housing_labels_train_y[:5].values)
print(f"Tree RMSE sebesar: {tree_rmse}")

[458300. 483800. 101700.  96100. 361800.]
[458300. 483800. 101700.  96100. 361800.]
Tree RMSE sebesar: 0.0
